In [ ]:
import os
import sys

sys.path.append(os.path.abspath(os.path.join("..")))
import re
import operator
from typing import Annotated, TypedDict, cast

import arxiv
from IPython.display import Image, Markdown, display
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    get_buffer_string,
)
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from pydantic import BaseModel, Field

from utils.models import Models, Trace, get_invoke_options, ModelName
from utils.search import TavilyClient, TinyFish, create_arxiv

os.environ["LANGGRAPH_ALLOWED_MSGPACK_MODULES"] = "__main__"


In [ ]:
model = Models()
trace = Trace()
invoke_options = get_invoke_options()
nvidia_model = model.create_nvidia_model()
tiny_fish = TinyFish()
tavily_client = TavilyClient()


In [23]:
arxiv_search = create_arxiv(search_query="langchain and AI", max_results=3)
arxiv_search


In [1]:
def context_text(context: list[str]):
    seen = {}
    numbered = []
    for chunk in context:
        for doc in chunk.split("\n\n---\n\n"):
            doc = doc.strip()
            m = re.search(r'href="([^"]+)"', doc)
            href = m.group(1) if m else doc[:80]
            if href not in seen:
                seen[href] = len(seen) + 1
                numbered.append(f"[{seen[href]}] {doc}")
    return numbered

### **_Create Analysts_**

In [ ]:
analysts_instructions = """You are tasked with creating a set of AI analyst personas(which resembles humans).Follow the instructions carefully
1. First review and understand the topic:{topic}
2.Read and examine the optionally provided human_analyst_feedback instructions and follow them as guide in creating the analysts:{human_analyst_feedback}
3.Determine the interesting themes based on the documents and/or instructions/feedback above
4.pick the top {max_analysts} themes
5.Assign each analyst affiliation in context of the theme and topic
6.Assign each analyst a theme
"""

In [ ]:
class Analyst(BaseModel):
    name: str = Field(description="Name of the analyst(should resemble human names)")
    role: str = Field(description="Role of the analyst in context of the topic")
    affiliation: str = Field(
        description="Primary affiliation of the analyst in context fo the topic"
    )
    description: str = Field(
        description="Description of the analyst focus,motives and concerns"
    )

    @property
    def persona(self) -> str:
        return f"Name:{self.name}\n Role:{self.role} \nAffiliation:{self.affiliation} \nDescription:{self.description}"


class Perspectives(BaseModel):
    """Comprehensive list of the analysts"""

    analysts: list[Analyst] = Field(
        description="Comprehensive list of the analysts with their roles and affiliations "
    )


class GeneralAnalystState(TypedDict):
    topic: str
    max_analysts: int
    human_analyst_feedback: str
    analysts: list[Analyst]

In [ ]:
def create_analysts(state: GeneralAnalystState):
    topic = state["topic"]
    max_analysts = state["max_analysts"]
    human_analyst_feedback = state.get("human_analyst_feedback", "")
    sys_msg = analysts_instructions.format(
        topic=topic,
        max_analysts=max_analysts,
        human_analyst_feedback=human_analyst_feedback,
    )
    structured_llm = nvidia_model.with_structured_output(Perspectives)
    response = structured_llm.invoke(
        [SystemMessage(sys_msg)]
        + [HumanMessage(content="Generate the set of analysts")],
        **invoke_options,
    )

    return {"analysts": response.analysts}  # pyright:ignore[reportAttributeAccessIssue]

In [ ]:
def human_feedback(state: GeneralAnalystState):
    """Dummy Node"""
    pass


def should_continue(state: GeneralAnalystState):
    human_analyst_feedback = state.get("human_analyst_feedback", "")
    if human_analyst_feedback:
        return "create_analysts"
    return END


In [ ]:
builder = StateGraph(GeneralAnalystState)
builder.add_node("create_analysts", create_analysts)
builder.add_node("human_feedback", human_feedback)
builder.add_edge(START, "create_analysts")
builder.add_edge("create_analysts", "human_feedback")
builder.add_conditional_edges(
    "human_feedback", should_continue, ["create_analysts", END]
)
graph = builder.compile(interrupt_before=["human_feedback"], checkpointer=MemorySaver())
display(Image(graph.get_graph().draw_mermaid_png()))

In [ ]:
thread: RunnableConfig = {"configurable": {"thread_id": "1"}}

In [ ]:
topic = "Health issues in women over 40 and use of peptides in them"
max_analysts = 3

In [ ]:
for event in graph.stream(
    {"topic": topic, "max_analysts": max_analysts}, thread, stream_mode="values"
):
    analysts = event.get("analysts", "")
    if analysts:
        for analyst in analysts:
            print(f"Name:{analyst.name}")
            print(f"Role:{analyst.role}")
            print(f"Affiliation:{analyst.affiliation}")
            print(f"Description:{analyst.description}")
            print("-" * 75)

In [ ]:
state = graph.get_state(thread)
state.next

In [ ]:
graph.update_state(
    thread,
    {"human_analyst_feedback": "Add an additional analyst to the list"},
    as_node="human_feedback",
)

In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    analysts = event.get("analysts", "")
    if analysts:
        for analyst in analysts:
            print(f"Name:{analyst.name}")
            print(f"Role:{analyst.role}")
            print(f"Affiliation:{analyst.affiliation}")
            print(f"Description:{analyst.description}")
            print("-" * 75)


In [ ]:
state = graph.get_state(thread)
state.next


In [ ]:
# graph.update_state(
#     thread,
#     {"max_analysts": 4},
#     as_node="human_feedback",
# )


In [ ]:
# state = graph.get_state(thread)
# state.next


In [ ]:
for event in graph.stream(None, thread, stream_mode="values"):
    analysts = event.get("analysts", "")
    if analysts:
        for analyst in analysts:
            print(f"Name:{analyst.name}")
            print(f"Role:{analyst.role}")
            print(f"Affiliation:{analyst.affiliation}")
            print(f"Description:{analyst.description}")
            print("-" * 75)


In [ ]:
for event in graph.stream(None, thread, stream_mode="updates"):
    print("---Node---")
    node_name = next(iter(event.keys()))
    print(node_name)

In [ ]:
graph.update_state(thread, {"human_analyst_feedback": None}, as_node="human_feedback")

In [ ]:
graph.get_state(thread).next

In [ ]:
final_state = graph.get_state(thread)
analysts = final_state.values.get("analysts")


In [ ]:
if analysts:
    for analyst in analysts:
        print(f"Name:{analyst.name}")
        print(f"Role:{analyst.role}")
        print(f"Affiliation:{analyst.affiliation}")
        print(f"Description:{analyst.description}")
        print("-" * 75)

### **_Interview_**

In [ ]:
interview_instructions = """You are an analyst tasked with interviewing an expert to learn about a specific topic
Your goal is to boil down to interesting and specific facts related to the topic 
1.Interesting:Insights that people find surprising and non-obvious
2.Specific: Insights that avoid non-generalities and include specific examples from the expert
Here is your topic to focus and the set of goals:{goals}
Start the interview by introducing yourself with name and affiliation that fits your persona
Continue asking questions to drill down and refine your understanding of the topic.
When you are satisfied with the understanding,complete the interview with:"Thank you so much for your help"
Remember to stay in the character as mentioned in the persona throughout the interview and never drift away from your persona
 """

In [ ]:
class InterviewState(MessagesState):
    max_num_turns: int
    context: Annotated[list, operator.add]
    interview: str
    sections: list
    analyst: Analyst


class SearchQuery(BaseModel):
    search_query: str

In [ ]:
def generate_questions(state: InterviewState):
    analyst = state["analyst"]
    messages = state["messages"]
    sys_msg = interview_instructions.format(goals=analyst.persona)
    question = nvidia_model.invoke(
        [SystemMessage(content=sys_msg)] + messages, **invoke_options
    )
    return {"messages": [question]}

In [ ]:
search_instructions = """You will be  given an conversation between an analyst and an expert
Your job is to generate a well structured query for use in web search and/or retrieval related to the conversation
First analyze the complete conversation.
Pay close attention to the final question posed by the analyst, 
Convert this final question based on the rest of the conversation as a well structured query for web search and/or retrieval.
"""

In [ ]:
def web_search(state: InterviewState):
    """Search docs from the web"""
    structured_llm = nvidia_model.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke(
        [SystemMessage(content=search_instructions)] + state["messages"],
        **invoke_options,
    )

    response = tavily_client.search(search_query.search_query)  # pyright:ignore[reportAttributeAccessIssue]

    results = [result for result in response["results"] if len(result["content"]) > 800]

    formatted_docs = "\n\n---\n\n".join(
        [
            f'<Document href="{doc["url"]}">{doc["content"]}</Document>'
            for doc in results
        ]
    )

    return {"context": [formatted_docs]}


In [ ]:
def arxiv_search(state: InterviewState):
    """Search docs from Arxiv"""

    structured_llm = nvidia_model.with_structured_output(SearchQuery)
    search_query = structured_llm.invoke(
        [SystemMessage(content=search_instructions)] + state["messages"],
    )
    client = arxiv.Client()
    results = client.results(arxiv.Search(search_query.search_query, max_results=3))  # pyright:ignore[reportAttributeAccessIssue]

    formatted_docs = "\n\n---\n\n".join(
        [f'<Document href="{doc.entry_id}">{doc.summary}</Document>' for doc in results]
    )

    return {"context": [formatted_docs]}

In [ ]:
answer_instructions = """You are an expert being interviewed by an analyst. 
Here is the analyst area of focus:{goals}"
Your job is to answer a question posed to you by the interviewer,
To answer a question, use this context:
{context}
when answering a question,follow these guidelines:
1.Use only the information provided in the context.
2.Do not introduce external information or make assumptions beyond what is explicitly mentioned in the context.
3.The context contain sources at topic of each individual Document.
4.Include these sources your answer next to any relevant statement.For example for source #1 use [1]
5. List your sources at the bottom of your answer. [1] Source 1, [2] Source 2,etc 
6.If the source is: <Document source="assistant/docs/llama3_1.pdf" page="7"/>' then just list: 
        [1] assistant/docs/llama3_1.pdf, page 7 
7.And skip the addition of brackets as well as the Document source preamble in your citation

"""

In [ ]:
def generate_answer(state: InterviewState):
    """Node to generate answers"""
    analyst = state["analyst"]
    messages = state["messages"]
    context = state["context"]

    sys_msg = answer_instructions.format(goals=analyst.persona, context=context)
    answer = nvidia_model.invoke(
        [SystemMessage(content=sys_msg)] + messages, **invoke_options
    )
    answer.name = "expert"

    return {"messages": [answer]}


def save_interview(state: InterviewState):
    """Save Interview"""
    messages = state["messages"]
    interview = get_buffer_string(messages)

    return {"interview": interview}


def route_messages(state: InterviewState, name: str = "expert"):
    """Route between question and answer"""
    messages = state["messages"]
    max_num_turns = state.get("max_num_turns", 2)
    num_responses = len(
        [m for m in messages if isinstance(m, AIMessage) and m.name == name]
    )
    if num_responses >= max_num_turns:
        return "save_interview"
    last_question = messages[-2]

    if "Thank you so much for your help" in last_question.content:
        return "save_interview"
    return "ask_question"

In [ ]:
section_writer_instructions = """You are an expert technical writer
Your job is to create short easily digestible section of a report based on the set of source documents.
1.Analyze the content of the source documents:
 - The name of each source document is at the start of the document,with the document tag. 
2. Create a report structure using markdown formatting 
 - use ## for section title 
 - Use ### for section sub-title headers 
3. Write the report following this structure 
    a. Title (## header)
    b. Summary (### header)
    c. Sources (### Header) 
4. Make your title engaging based upon the focus area of the analyst: {focus}
5.For summary section:
    - set up summary with general background /context related to the focus area of the analyst 
    - Emphasize what is novel,interesting or surprising about insights gathered from the interview
    - Create a numbered list of source documents, as you use them 
    - Aim for maximum 600 words 
    - Use numbered sources in your report(e.g., [1],[2] based on information from source documents)
6. In the sources section:
- Include all sources used in your report 
- Provide full links to relevant websites or specific document paths 
- Separate each source by a newline.Use two spaces at the end of each line to create a newline in Markdown 
- It will look like:
    ### Sources 
    [1] Link or Document name 
    [2] Link or Document name 
7. Be sure to combine sources.For example this is not correct:
    [3] https://ai.meta.com/blog/meta-llama-3-1/
    [4] https://ai.meta.com/blog/meta-llama-3-1/

    There should never be redundant sources. It should simply be:
      [3] https://ai.meta.com/blog/meta-llama-3-1/
8. Final review:
    - Ensure the report follows the required structure 
    - Include no preamble before the title of the report 
    - Check all the guidelines have been followed

"""


In [ ]:
def write_section(state: InterviewState):
    """Node to answer a question"""
    interview = state["interview"]  # INFO Is this unused variable??
    context = state["context"][]
    analyst = state["analyst"]

    sys_message = section_writer_instructions.format(focus=analyst.description)
    section = nvidia_model.invoke(
        [SystemMessage(content=sys_message)]
        + [HumanMessage(content=f"use this source to your section :{context}")],
        **invoke_options,
    )
    return {"sections": [section.content]}


In [ ]:
interview_builder = StateGraph(InterviewState)
interview_builder.add_node("ask_question", generate_questions)
interview_builder.add_node("search_web", web_search)
interview_builder.add_node("search_arxiv", arxiv_search)
interview_builder.add_node("answer_question", generate_answer)
interview_builder.add_node("save_interview", save_interview)
interview_builder.add_node("write_section", write_section)

interview_builder.add_edge(START, "ask_question")
interview_builder.add_edge("ask_question", "search_web")
interview_builder.add_edge("ask_question", "search_arxiv")
interview_builder.add_edge("search_web", "answer_question")
interview_builder.add_edge("search_arxiv", "answer_question")
interview_builder.add_conditional_edges(
    "answer_question", route_messages, ["ask_question", "save_interview"]
)
interview_builder.add_edge("save_interview", "write_section")
interview_builder.add_edge("write_section", END)

interview_graph = interview_builder.compile(checkpointer=MemorySaver()).with_config(
    run_name="Conduct Interview"
)
display(Image(interview_graph.get_graph().draw_mermaid_png()))

In [ ]:
analysts[0]

In [ ]:
messages = [HumanMessage(f"So you said you were writing an article on {topic}?")]
thread: RunnableConfig = {"configurable": {"thread_id": "1"}}
interview = interview_graph.invoke(
    {"analyst": analysts[0], "messages": messages, "max_num_turns": 2}, thread
)
Markdown(interview["sections"][0])

In [ ]:
print(interview["sections"][0])

In [ ]:
interview["sections"]